# QUTCC Checkpoint Explorer

Load a trained QUTCC quantile model and interactively explore:

- **Reconstruction** (median prediction, q = 0.5)
- **UQ map** (predicted interval width, a per-pixel uncertainty heatmap)
- **A "hallucination"** — a region where the reconstruction confidently diverges from
  ground truth even though the predicted uncertainty interval doesn't cover it
- **Per-pixel PDF maps** — the full predictive distribution at individual pixels,
  reconstructed from many quantile levels (same idea as `QUTCC_eval.get_quantile_outputs`)

## Setup

1. Put a trained checkpoint (an `epoch*.pth` file containing a `model_state_dict`, saved
   by `train.py` for a `unet_quantile` run) somewhere in your Google Drive.
2. Put a sample `(noisy, clean)` image pair in Drive too, as either:
   - two `.npy` files, each shape `(H, W)` or `(1, H, W)`, **or**
   - a single `.pt` file containing a dict `{"noisy": tensor, "clean": tensor}`.
   A quick way to produce one: pull a single example out of any of the dataloaders in
   `datasets/` and `torch.save({"noisy": noisy[0], "clean": clean[0]}, "sample.pt")`.
3. Fill in the paths in the **Config** cell below and run top to bottom.

This notebook only loads the quantile (`unet_quantile`) model — it doesn't need the
`im2im` or `unet_im2im` checkpoints or the `best_runs.json` / experiment-folder
machinery that `analysis.py` uses for the paper's full evaluation sweep.

## 1. Environment setup

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Path to your local clone of the QUTCC repo (if running in Colab, this should be
# wherever you cloned/copied the repo inside Drive, e.g. "/content/drive/MyDrive/QUTCC").
REPO_DIR = Path(".").resolve()
if not (REPO_DIR / "models").exists():
    # fall back: assume this notebook lives in <repo>/notebooks/
    REPO_DIR = Path("..").resolve()

assert (REPO_DIR / "models" / "checkpointing.py").exists(), (
    f"Couldn't find the QUTCC repo at {REPO_DIR} -- set REPO_DIR manually above."
)
sys.path.insert(0, str(REPO_DIR))
print(f"Using repo at: {REPO_DIR}")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from models.checkpointing import create_model, load_model_state
from QUTCC_eval import get_quantile_outputs, extract_sorted_pdf, plot_pdf

plt.rcParams["figure.facecolor"] = "white"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Config

Edit these to point at your uploaded checkpoint and sample image.

In [ ]:
# --- Required: path to your checkpoint ---
CHECKPOINT_PATH = "/content/drive/MyDrive/QUTCC/checkpoints/epoch30.pth"  # <-- EDIT ME

# --- Required: path to your sample image(s) ---
# Option A: a single .pt file with {"noisy": tensor, "clean": tensor}
SAMPLE_PT_PATH = "/content/drive/MyDrive/QUTCC/sample.pt"  # <-- EDIT ME (or set to None)
# Option B: two .npy files (used only if SAMPLE_PT_PATH is None)
NOISY_NPY_PATH = "/content/drive/MyDrive/QUTCC/noisy.npy"
CLEAN_NPY_PATH = "/content/drive/MyDrive/QUTCC/clean.npy"

# Model config -- match whatever the checkpoint was trained with
IN_CHANNELS = 1

# Calibrated quantile bounds for the UQ map. 0.05/0.95 are the uncalibrated defaults
# used at training time (see models/checkpointing.py); if you've run calibration_sweep.py
# for this checkpoint, use the lower_q/upper_q it found instead for a conformal interval.
LOWER_Q = 0.05
UPPER_Q = 0.95

## 3. Load the model

In [ ]:
model = create_model(net="unet_quantile", device=DEVICE, in_channels=IN_CHANNELS)
model = load_model_state(model, CHECKPOINT_PATH, DEVICE)
model.eval()
print("Model loaded.")

## 4. Load a sample image

In [ ]:
def _to_1chw(x) -> torch.Tensor:
    x = torch.as_tensor(x, dtype=torch.float32)
    while x.ndim < 3:
        x = x.unsqueeze(0)
    if x.ndim == 3:
        x = x.unsqueeze(0)  # -> [1, C, H, W]
    return x

if SAMPLE_PT_PATH is not None and Path(SAMPLE_PT_PATH).exists():
    sample = torch.load(SAMPLE_PT_PATH, map_location="cpu")
    noisy, clean = _to_1chw(sample["noisy"]), _to_1chw(sample["clean"])
else:
    noisy, clean = _to_1chw(np.load(NOISY_NPY_PATH)), _to_1chw(np.load(CLEAN_NPY_PATH))

noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)
print(f"noisy: {noisy.shape}, clean: {clean.shape}")

## 5. Run inference: reconstruction + UQ map

The quantile model takes the quantile level as a conditioning input (reusing the
diffusion-UNet's timestep embedding), so getting the median, lower, and upper bound
each takes a separate forward pass -- see `QUTCC_eval.plot_visualization` for the
same pattern.

In [ ]:
def predict_quantile(x: torch.Tensor, q: float) -> torch.Tensor:
    q_tensor = torch.full((x.shape[0],), q, device=DEVICE, dtype=torch.float32)
    with torch.no_grad():
        return model(x, q_tensor)

median_pred = predict_quantile(noisy, 0.5)
lower_pred  = predict_quantile(noisy, LOWER_Q)
upper_pred  = predict_quantile(noisy, UPPER_Q)

uq_map   = (upper_pred - lower_pred).clamp(min=0)          # interval width, per pixel
residual = (median_pred - clean).abs()                      # |reconstruction - ground truth|
print(f"Mean UQ width: {uq_map.mean().item():.4f}   Mean residual: {residual.mean().item():.4f}")

In [ ]:
def show(ax, img, title, cmap="gray", vmax=None):
    im = ax.imshow(img.squeeze().cpu().numpy(), cmap=cmap, vmax=vmax)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    return im

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
show(axes[0], noisy[0], "Noisy input")
show(axes[1], clean[0], "Ground truth")
show(axes[2], median_pred[0], "Reconstruction (q=0.5)")
im = show(axes[3], residual[0], "Residual |pred - gt|", cmap="inferno")
fig.colorbar(im, ax=axes[3], fraction=0.046)
im = show(axes[4], uq_map[0], "UQ map (interval width)", cmap="viridis")
fig.colorbar(im, ax=axes[4], fraction=0.046)
fig.tight_layout()
plt.show()

## 6. Find a "hallucination"

A hallucination here means: the reconstruction is confidently *wrong* -- the ground
truth value falls outside the model's own predicted `[lower, upper]` interval. That's
different from "high uncertainty" (a wide interval that correctly contains the truth);
it's the model being both wrong *and* overconfident at that pixel.

In [ ]:
missed_mask = (clean < lower_pred) | (clean > upper_pred)  # True where gt fell outside the interval
print(f"Fraction of pixels where ground truth fell outside the predicted interval: "
      f"{missed_mask.float().mean().item():.4%}")

# Pick the worst offender: largest residual among the missed (hallucinated) pixels.
masked_residual = residual.clone()
masked_residual[~missed_mask] = -1
flat_idx = masked_residual[0, 0].argmax()
hallu_row, hallu_col = divmod(flat_idx.item(), masked_residual.shape[-1])
print(f"Largest hallucination at pixel (row={hallu_row}, col={hallu_col}), "
      f"residual={residual[0, 0, hallu_row, hallu_col].item():.4f}")

# For comparison, pick a well-behaved pixel: small residual, inside the interval.
calibrated_residual = residual.clone()
calibrated_residual[missed_mask] = 1e9
flat_idx_good = calibrated_residual[0, 0].argmin()
good_row, good_col = divmod(flat_idx_good.item(), calibrated_residual.shape[-1])
print(f"Well-calibrated pixel at (row={good_row}, col={good_col}), "
      f"residual={residual[0, 0, good_row, good_col].item():.4f}")

In [ ]:
zoom = 40
r0, c0 = max(hallu_row - zoom // 2, 0), max(hallu_col - zoom // 2, 0)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
show(axes[0], clean[0], "Ground truth (full)")
axes[0].add_patch(plt.Rectangle((c0, r0), zoom, zoom, edgecolor="red", facecolor="none", lw=1.5))
axes[0].scatter([hallu_col], [hallu_row], c="red", s=15)

crop = lambda img: img[0, 0, r0:r0 + zoom, c0:c0 + zoom].cpu().numpy()
axes[1].imshow(crop(clean), cmap="gray"); axes[1].set_title("Ground truth (zoom)"); axes[1].axis("off")
axes[2].imshow(crop(median_pred), cmap="gray"); axes[2].set_title("Reconstruction (zoom)"); axes[2].axis("off")
im = axes[3].imshow(crop(missed_mask.float()), cmap="Reds", vmin=0, vmax=1)
axes[3].set_title("Hallucination mask\n(gt outside predicted interval)"); axes[3].axis("off")
fig.tight_layout()
plt.show()

## 7. Per-pixel PDF maps

`get_quantile_outputs` runs the model at many quantile levels and turns the quantile
function into a density estimate via `1 / dQ/dp` (same as `QUTCC_eval.__init__`).
Comparing the PDF at the hallucinated pixel against a well-calibrated pixel shows
*why* the interval missed: the predicted density can be narrow and off-center,
while a well-behaved pixel's density is centered on the true value.

In [ ]:
quantile_levels = np.linspace(0.02, 0.98, 25)
quantile_preds, pdf_est = get_quantile_outputs(model, noisy, quantile_levels, DEVICE)
print(f"quantile_preds: {quantile_preds.shape}  (levels, H, W)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

s_bad, p_bad = extract_sorted_pdf(quantile_preds, pdf_est, hallu_row, hallu_col)
s_good, p_good = extract_sorted_pdf(quantile_preds, pdf_est, good_row, good_col)

plot_pdf(ax, s_bad, p_bad, color="crimson", label=f"Hallucinated pixel ({hallu_row},{hallu_col})")
plot_pdf(ax, s_good, p_good, color="steelblue", label=f"Well-calibrated pixel ({good_row},{good_col})")

ax.axvline(clean[0, 0, hallu_row, hallu_col].item(), color="crimson", ls="--", lw=1,
           label="Ground truth @ hallucinated pixel")
ax.axvline(clean[0, 0, good_row, good_col].item(), color="steelblue", ls="--", lw=1,
           label="Ground truth @ calibrated pixel")

ax.set_xlabel("Pixel intensity")
ax.set_ylabel("Estimated density")
ax.legend(fontsize=8)
ax.set_title("Per-pixel predictive PDF: hallucination vs. well-calibrated pixel")
fig.tight_layout()
plt.show()

## Notes

- To use a different task's checkpoint, just change `CHECKPOINT_PATH` and `IN_CHANNELS`
  (and `LOWER_Q`/`UPPER_Q` if you've calibrated it) -- no other code needs to change,
  since `unet_quantile` always takes `(image, quantile_level)` as input regardless of task.
- If you've calibrated this checkpoint with `calibration_sweep.py`, the resulting
  `lower_q`/`upper_q` give a conformal interval instead of the raw 0.05/0.95 training
  quantiles -- plug those in for a tighter, coverage-guaranteed UQ map.
- For a full batch/dataset sweep instead of one image, loop section 5-7 over a
  `DataLoader` the same way `evaluation.get_test_intervals_missed` does.